# LeetCode 3769 — Sort Integers by Binary Reflection bằng Python

Link đề: https://leetcode.com/problems/sort-integers-by-binary-reflection/ · Độ khó: **Easy** · Chủ đề: Array, Sorting, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def sortByReflection(self, nums: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm (`sortByReflection`) đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **phản chiếu nhị phân** (binary reflection) của số nguyên dương là số thu được khi đảo các chữ số nhị phân rồi đọc lại thành số thập phân; sắp xếp mảng tăng dần theo phản chiếu, hòa thì số nhỏ hơn trước.
2. Biết tính phản chiếu theo hai cách: bằng chuỗi (`bin`, `[::-1]`, `int(..., 2)`) và bằng vòng lặp bit (`<<`, `&`, `>>`).
3. Thành thạo `sorted(..., key=...)` với **khóa là tuple** `(phản chiếu, giá trị)` để xử lý hòa, và hiểu vì sao sắp xếp Python là **ổn định**.
4. Viết 4 cách: sắp xếp thủ công (insertion sort) bằng so sánh khóa, `sorted` với khóa chuỗi, `sorted` với khóa bit, và `cmp_to_key`; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên tiêu chí hòa, giữ độ rộng 32 bit khi đảo, so sánh chuỗi đảo theo từ điển, và đảo chữ số **thập phân** thay vì nhị phân.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums`. **Phản chiếu nhị phân** của một số nguyên dương là giá trị thập phân của chuỗi nhị phân của nó **sau khi đảo ngược** (bỏ các số `0` ở đầu kết quả). Ví dụ `8 = 1000` -> đảo `0001` -> phản chiếu `1`.

Sắp xếp `nums` theo thứ tự **tăng dần của phản chiếu**. Nếu hai số có cùng phản chiếu thì số **nhỏ hơn** đứng trước. Trả về mảng đã sắp xếp.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[4,5,4]` | `[4,4,5]` | `4 = 100 -> 001 = 1`, `5 = 101 -> 101 = 5`; phản chiếu `1, 5, 1` |
| 2 | `[3,6,5,8]` | `[8,3,6,5]` | phản chiếu: `3 -> 3`, `6 = 110 -> 011 = 3`, `5 -> 5`, `8 -> 1`; `3` và `6` hòa nên `3` trước |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 10^9`.

In [1]:
def reflect(x):
    return int(bin(x)[2:][::-1], 2)

for nums in [[4, 5, 4], [3, 6, 5, 8]]:
    print(nums, "phản chiếu:", [reflect(x) for x in nums])
    print("   kết quả:", sorted(nums, key=lambda x: (reflect(x), x)))

[4, 5, 4] phản chiếu: [1, 5, 1]
   kết quả: [4, 4, 5]
[3, 6, 5, 8] phản chiếu: [3, 3, 5, 1]
   kết quả: [8, 3, 6, 5]


## 3. Kiến thức nền

### 3.1. Chuỗi nhị phân, `bin`, đảo chuỗi và `int(s, 2)`

`bin(x)[2:]` là chuỗi nhị phân (bỏ `'0b'`). `s[::-1]` đảo chuỗi. `int(s, 2)` đọc chuỗi nhị phân thành số, **tự bỏ qua các số `0` đứng đầu**. Vì vậy `int('0001', 2) = 1` đúng ý "bỏ `0` đầu" của đề.

In [2]:
x = 8
s = bin(x)[2:]
print(s, s[::-1], int(s[::-1], 2))
print(int('0001', 2), int('011', 2), int('101', 2))

1000 0001 1
1 3 5


### 3.2. Tính phản chiếu bằng thao tác bit

Có thể đảo bit không cần chuỗi: lấy bit thấp nhất của `x` bằng `x & 1`, đẩy vào kết quả `r = (r << 1) | (x & 1)`, rồi `x >>= 1`; lặp đến khi `x = 0`. Bit thấp nhất của `x` sẽ trở thành bit **cao nhất** của kết quả; số `0` thừa ở cuối `x` (ví dụ `1000`) lần lượt đẩy `0` vào `r` khi `r = 0` nên vô hại.

In [3]:
def reflect_bits(x):
    r = 0
    while x:
        r = (r << 1) | (x & 1)
        x >>= 1
    return r

print([reflect_bits(v) for v in [4, 5, 3, 6, 8, 1]])
print(all(reflect_bits(v) == reflect(v) for v in range(1, 5000)))

[1, 5, 3, 3, 1, 1]
True


### 3.3. `sorted`, `key=` và khóa là tuple

`sorted(lst, key=f)` sắp xếp theo giá trị `f(x)`. Nếu `f` trả về **tuple** `(a, b)`, Python so sánh `a` trước, hòa thì so `b`... Đây là cách chuẩn để thể hiện "ưu tiên 1 rồi ưu tiên 2".

In [4]:
words = ["bb", "a", "ccc", "aa", "b"]
print(sorted(words, key=lambda w: (len(w), w)))        # độ dài rồi theo từ điển
print((1, 5) < (1, 7), (2, 0) < (1, 9))

['a', 'b', 'aa', 'bb', 'ccc']
True False


### 3.4. Sắp xếp ổn định (stable)

`sorted` của Python là **ổn định**: các phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu. Vì vậy nếu chỉ sắp theo `reflect(x)` thì các số hòa nhau sẽ ở thứ tự **ban đầu** chứ không phải tăng dần; ta cần khóa thứ hai `x` để ép tiêu chí hòa của đề.

In [5]:
nums = [6, 3]                                  # cùng phản chiếu 3
print(sorted(nums, key=reflect))               # ổn định: giữ [6, 3]
print(sorted(nums, key=lambda x: (reflect(x), x)))     # đúng đề: [3, 6]

[6, 3]
[3, 6]


### 3.5. `functools.cmp_to_key`

Ngoài khóa, có thể sắp bằng **hàm so sánh** `cmp(a, b)` trả số âm/0/dương; `cmp_to_key(cmp)` biến nó thành khóa. Hữu ích khi tiêu chí khó biểu diễn bằng một khóa, nhưng chậm hơn và dễ sai dấu.

In [6]:
from functools import cmp_to_key

def cmp(a, b):
    ra, rb = reflect(a), reflect(b)
    if ra != rb:
        return -1 if ra < rb else 1
    return -1 if a < b else (1 if a > b else 0)

print(sorted([3, 6, 5, 8], key=cmp_to_key(cmp)))

[8, 3, 6, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đây là bài "sắp xếp theo một đại lượng tự định nghĩa". Cứ chia hai bước:

1. Với mỗi số, **tính đại lượng** (phản chiếu).
2. **Sắp xếp** theo đại lượng đó, hòa thì theo giá trị gốc.

Tính tay ví dụ 2 `[3, 6, 5, 8]`:

```
 số   nhị phân   đảo     phản chiếu
  3     11        11         3
  6     110       011        3
  5     101       101        5
  8     1000      0001       1
```

Sắp theo (phản chiếu, giá trị): `(1, 8), (3, 3), (3, 6), (5, 5)` -> `[8, 3, 6, 5]`.

Nhận xét hay: số chẵn có chữ số `0` ở cuối nhị phân, đảo lên thành `0` ở đầu bị bỏ, nên **phản chiếu của `x` bằng phản chiếu của `x` đã bỏ hết các bit `0` ở cuối** (`x / 2^t` với `t` là số bit `0` cuối). Do đó nhiều số khác nhau có cùng phản chiếu (`3` và `6`, `12`, `24`, ...), và tiêu chí hòa **thực sự được dùng**.

### Bảng phản chiếu của `1..16`

Hãy tự in và quan sát: các số chẵn trùng phản chiếu với "nửa" của chúng.

In [7]:
print(f"{'x':>3} {'bin':>6} {'đảo':>6} {'phản chiếu':>10}")
for x in range(1, 17):
    s = bin(x)[2:]
    print(f"{x:>3} {s:>6} {s[::-1]:>6} {int(s[::-1], 2):>10}")
assert all(reflect(2 * x) == reflect(x) for x in range(1, 5000))
print("phản chiếu(2x) == phản chiếu(x)")

  x    bin    đảo phản chiếu
  1      1      1          1
  2     10     01          1
  3     11     11          3
  4    100    001          1
  5    101    101          5
  6    110    011          3
  7    111    111          7
  8   1000   0001          1
  9   1001   1001          9
 10   1010   0101          5
 11   1011   1101         13
 12   1100   0011          3
 13   1101   1011         11
 14   1110   0111          7
 15   1111   1111         15
 16  10000  00001          1
phản chiếu(2x) == phản chiếu(x)


## 5. Cách 1 — Brute Force / baseline

Tự cài **insertion sort**: duyệt từng phần tử, dịch các phần tử lớn hơn sang phải (theo so sánh `(phản chiếu, giá trị)`) rồi chèn vào chỗ đúng. Không dùng `sorted`.

Nút thắt: `O(n^2)` so sánh, mỗi lần so sánh lại phải tính phản chiếu cho hai số (`O(log x)`), tức tính đi tính lại cùng một giá trị. Ta có thể tính phản chiếu **một lần** rồi dùng làm khóa.

In [8]:
class SolutionBrute:
    def sortByReflection(self, nums):
        def ref(x):
            r = 0
            while x:
                r = (r << 1) | (x & 1)
                x >>= 1
            return r

        a = list(nums)                                   # không sửa đầu vào
        for i in range(1, len(a)):
            cur = a[i]
            j = i - 1
            while j >= 0 and (ref(a[j]), a[j]) > (ref(cur), cur):
                a[j + 1] = a[j]
                j -= 1
            a[j + 1] = cur
        return a

print(SolutionBrute().sortByReflection([4, 5, 4]), SolutionBrute().sortByReflection([3, 6, 5, 8]))

[4, 4, 5] [8, 3, 6, 5]


**Độ phức tạp:** Time `O(n^2 log V)` (với `V` là giá trị lớn nhất, mỗi so sánh tính phản chiếu), Space `O(n)` cho bản sao.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** biểu diễn tiêu chí của đề **bằng một khóa duy nhất** là tuple `(phản chiếu, giá trị)`, rồi để `sorted` làm phần còn lại với `O(n log n)` so sánh và mỗi phản chiếu chỉ tính **một lần**.

**Quy tắc "nếu… thì…":**

- Nếu hai số có phản chiếu khác nhau -> số có phản chiếu nhỏ hơn đứng trước.
- Nếu phản chiếu bằng nhau -> số nhỏ hơn (theo giá trị gốc) đứng trước.
- Tuple `(a, b)` trong Python đã cài đúng quy tắc "so `a` trước, hòa thì so `b`".

**Invariant (insertion sort):** sau khi xử lý `i` phần tử đầu, đoạn `a[0..i]` đã **đúng thứ tự** theo khóa `(phản chiếu, giá trị)`.

In [9]:
# Kiểm chứng: sắp theo khóa tuple cho cùng kết quả với insertion sort
import random
random.seed(2)
for _ in range(300):
    nums = [random.randint(1, 10 ** 9) for _ in range(random.randint(1, 30))]
    assert sorted(nums, key=lambda x: (reflect(x), x)) == SolutionBrute().sortByReflection(nums)
print("khóa tuple khớp insertion sort")

khóa tuple khớp insertion sort


## 7. Các cách tiếp theo

### Cách 2 — `sorted` với khóa tính bằng chuỗi

Viết `reflect(x) = int(bin(x)[:1:-1], 2)` (lát cắt đảo từ cuối đến ký tự thứ 2 tức bỏ `'0b'`) hoặc dễ đọc hơn `int(bin(x)[2:][::-1], 2)`; rồi `sorted(nums, key=lambda x: (reflect(x), x))`. Ngắn, rõ, là lời giải chuẩn.

In [10]:
class SolutionKeyString:
    def sortByReflection(self, nums):
        return sorted(nums, key=lambda x: (int(bin(x)[2:][::-1], 2), x))

print(SolutionKeyString().sortByReflection([4, 5, 4]), SolutionKeyString().sortByReflection([3, 6, 5, 8]))

[4, 4, 5] [8, 3, 6, 5]


**Độ phức tạp:** Time `O(n log n + n log V)`, Space `O(n)`.

### Cách 3 — `sorted` với khóa tính bằng thao tác bit

Cùng khung nhưng tự cài hàm `reflect_bits` bằng vòng `while x`. Phù hợp khi muốn tránh chuyển sang chuỗi (hữu ích khi luyện bit trick) và hiểu rõ phép đảo.

In [11]:
class SolutionKeyBits:
    def sortByReflection(self, nums):
        def reflect_bits(x):
            r = 0
            while x:
                r = (r << 1) | (x & 1)
                x >>= 1
            return r
        return sorted(nums, key=lambda x: (reflect_bits(x), x))

print(SolutionKeyBits().sortByReflection([4, 5, 4]), SolutionKeyBits().sortByReflection([3, 6, 5, 8]))

[4, 4, 5] [8, 3, 6, 5]


**Độ phức tạp:** Time `O(n log n + n log V)`, Space `O(n)`.

### Cách 4 — Hàm so sánh với `cmp_to_key`

Viết `cmp(a, b)` so sánh phản chiếu trước rồi giá trị; bọc bằng `cmp_to_key`. Linh hoạt nhưng mỗi lần so sánh tính lại phản chiếu nên chậm hơn (và dễ sai dấu); ở đây ta lưu sẵn bảng phản chiếu bằng dict để tránh tính lại.

In [12]:
from functools import cmp_to_key

class SolutionCmp:
    def sortByReflection(self, nums):
        ref = {x: int(bin(x)[2:][::-1], 2) for x in set(nums)}      # tính mỗi giá trị một lần

        def cmp(a, b):
            if ref[a] != ref[b]:
                return -1 if ref[a] < ref[b] else 1
            return (a > b) - (a < b)                                 # -1, 0, 1

        return sorted(nums, key=cmp_to_key(cmp))

print(SolutionCmp().sortByReflection([4, 5, 4]), SolutionCmp().sortByReflection([3, 6, 5, 8]))

[4, 4, 5] [8, 3, 6, 5]


**Độ phức tạp:** Time `O(n log n)` so sánh + `O(n log V)` tính phản chiếu, Space `O(n)`.

## 8. TRACE từng bước

Trace insertion sort trên `[3, 6, 5, 8]` với khóa `(phản chiếu, giá trị)`: `3 -> (3, 3)`, `6 -> (3, 6)`, `5 -> (5, 5)`, `8 -> (1, 8)`.

| i | phần tử chèn | khóa | đoạn đã sắp xếp trước | sau khi chèn |
|---|---|---|---|---|
| 1 | 6 | (3,6) | [3] (khóa (3,3)) | [3, 6] |
| 2 | 5 | (5,5) | [3, 6] | [3, 6, 5] |
| 3 | 8 | (1,8) | [3, 6, 5] | [8, 3, 6, 5] |

Phần tử `8` có khóa nhỏ nhất nên dịch cả ba phần tử sang phải rồi lên đầu.

In [13]:
def trace_insertion(nums):
    key = lambda x: (reflect(x), x)
    a = list(nums)
    print("khóa:", [(x, key(x)) for x in a])
    for i in range(1, len(a)):
        cur, j = a[i], i - 1
        while j >= 0 and key(a[j]) > key(cur):
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = cur
        print(f"  i={i} chèn {cur} (khóa {key(cur)}) -> {a}")
    return a

assert trace_insertion([3, 6, 5, 8]) == [8, 3, 6, 5]
assert trace_insertion([4, 5, 4]) == [4, 4, 5]

khóa: [(3, (3, 3)), (6, (3, 6)), (5, (5, 5)), (8, (1, 8))]
  i=1 chèn 6 (khóa (3, 6)) -> [3, 6, 5, 8]
  i=2 chèn 5 (khóa (5, 5)) -> [3, 6, 5, 8]
  i=3 chèn 8 (khóa (1, 8)) -> [8, 3, 6, 5]
khóa: [(4, (1, 4)), (5, (5, 5)), (4, (1, 4))]
  i=1 chèn 5 (khóa (5, 5)) -> [4, 5, 4]
  i=2 chèn 4 (khóa (1, 4)) -> [4, 4, 5]


In [14]:
def trace_key_sort(nums):
    rows = [(reflect(x), x) for x in nums]
    print("cặp (phản chiếu, giá trị):", rows)
    rows.sort()
    print("sau sort:", rows)
    return [x for _, x in rows]

assert trace_key_sort([3, 6, 5, 8]) == [8, 3, 6, 5]

cặp (phản chiếu, giá trị): [(3, 3), (3, 6), (5, 5), (1, 8)]
sau sort: [(1, 8), (3, 3), (3, 6), (5, 5)]


## 9. Code nộp LeetCode

In [15]:
from typing import List, Optional

class Solution:
    def sortByReflection(self, nums: List[int]) -> List[int]:
        def reflect(x: int) -> int:
            return int(bin(x)[2:][::-1], 2)       # đảo chuỗi nhị phân, int() tự bỏ 0 đứng đầu

        return sorted(nums, key=lambda x: (reflect(x), x))

## 10. Vì sao thuật toán đúng

- **Khóa biểu diễn đúng đề:** đề yêu cầu thứ tự theo phản chiếu tăng dần, hòa thì giá trị tăng dần. So sánh tuple `(phản chiếu, giá trị)` theo thứ tự từ điển chính là quy tắc đó: so phản chiếu trước, hòa thì so giá trị.
- **Phản chiếu đúng:** `bin(x)[2:]` là biểu diễn nhị phân không có `0` đứng đầu; đảo chuỗi rồi `int(..., 2)` đọc lại thành số, tự bỏ các `0` đầu (chính là các `0` cuối của `x`).
- **Sắp xếp đúng:** `sorted` sắp tăng dần theo khóa; hai phần tử cùng khóa là hai số **bằng nhau** (vì khóa chứa giá trị gốc), nên thứ tự giữa chúng không ảnh hưởng đến kết quả. Không phụ thuộc vào tính ổn định.
- **Insertion sort:** invariant "đoạn đã xử lý luôn được sắp" được giữ vì ta chèn phần tử mới vào đúng vị trí; kết thúc khi xử lý hết mảng.
- **Cách `cmp_to_key`:** hàm `cmp` là một thứ tự toàn phần nhất quán (so phản chiếu rồi giá trị) nên cho cùng kết quả.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Insertion sort thủ công | `O(n^2 log V)` | `O(n)` |
| `sorted` + khóa chuỗi | `O(n log n + n log V)` | `O(n)` |
| `sorted` + khóa bit | `O(n log n + n log V)` | `O(n)` |
| `sorted` + `cmp_to_key` | `O(n log n + n log V)` | `O(n)` |

`n = len(nums) <= 100`, `V <= 10^9` nên `log V <= 30` (độ dài nhị phân). Phần sắp xếp chiếm `O(n log n)`, phần tính khóa chiếm `O(n log V)`.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `nums[i] = 1`, `10^9`, nhiều giá trị trùng, nhiều số cùng phản chiếu) và so sánh **tất cả cách** trên nhiều input hợp lệ; thêm kiểm tra tính chất: đầu ra là hoán vị của đầu vào và khóa không giảm.

In [16]:
impls = [SolutionBrute(), SolutionKeyString(), SolutionKeyBits(), SolutionCmp(), Solution()]

cases = [
    ([4, 5, 4], [4, 4, 5]), ([3, 6, 5, 8], [8, 3, 6, 5]),
    ([1], [1]), ([10 ** 9], [10 ** 9]), ([6, 3], [3, 6]), ([12, 6, 3, 24], [3, 6, 12, 24]),
    ([5, 3], [3, 5]), ([7, 9], [7, 9]), ([9, 7], [7, 9]),
    ([8, 4, 2, 1], [1, 2, 4, 8]), ([2, 2, 2], [2, 2, 2]),
]
for nums, exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= 10 ** 9 for v in nums)
    for impl in impls:
        got = impl.sortByReflection(list(nums))
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 11 ca


In [17]:
import random
random.seed(3769)
def oracle(nums):
    return sorted(nums, key=lambda x: (reflect(x), x))

for _ in range(600):
    n = random.randint(1, 100)
    mode = random.random()
    if mode < 0.4:
        nums = [random.randint(1, 10 ** 9) for _ in range(n)]
    elif mode < 0.7:
        nums = [random.randint(1, 64) for _ in range(n)]                     # nhiều trùng phản chiếu
    else:
        base = [random.randint(1, 1000) for _ in range(n)]
        nums = [min(10 ** 9, b << random.randint(0, 20)) for b in base]      # nhân 2^t giữ phản chiếu
    exp = oracle(nums)
    original = list(nums)
    outs = [impl.sortByReflection(list(nums)) for impl in impls]
    assert all(o == exp for o in outs), (nums, outs[0], exp)
    assert sorted(exp) == sorted(original)                                     # hoán vị
    keys = [(reflect(x), x) for x in exp]
    assert keys == sorted(keys)                                                # khóa không giảm
print("600 test ngẫu nhiên (seed=3769): 5 cách giống oracle, là hoán vị, khóa không giảm")

600 test ngẫu nhiên (seed=3769): 5 cách giống oracle, là hoán vị, khóa không giảm


In [18]:
# Đầu vào không bị sửa và kết quả là danh sách mới
nums = [3, 6, 5, 8]
copy = list(nums)
out = Solution().sortByReflection(nums)
assert nums == copy and out is not nums
big = [10 ** 9 - i for i in range(100)]
assert Solution().sortByReflection(big) == SolutionBrute().sortByReflection(big)
print("không sửa đầu vào, n = 100 OK")

không sửa đầu vào, n = 100 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên tiêu chí hòa (chỉ sắp theo phản chiếu)

Sắp xếp ổn định giữ thứ tự ban đầu của các phần tử hòa nhau; đề lại yêu cầu **số nhỏ hơn trước**.

In [19]:
def wrong_no_tiebreak(nums):
    return sorted(nums, key=lambda x: int(bin(x)[2:][::-1], 2))       # SAI: thiếu khóa thứ hai

print(wrong_no_tiebreak([6, 3]), "!=", Solution().sortByReflection([6, 3]))
assert wrong_no_tiebreak([6, 3]) == [6, 3] and Solution().sortByReflection([6, 3]) == [3, 6]

[6, 3] != [3, 6]


### Lỗi 2: đảo bit trong độ rộng cố định 32 bit

Khi đảo cả các `0` đứng đầu của số 32 bit, phản chiếu thành `reflect(x) * 2^(32 - L)`; hệ số này phụ thuộc độ dài `L` nên **thay đổi thứ tự**.

In [20]:
def wrong_32bit(nums):
    def r32(x):
        return int(format(x, '032b')[::-1], 2)                         # SAI: giữ 0 đầu
    return sorted(nums, key=lambda x: (r32(x), x))

print(wrong_32bit([3, 5]), "!=", Solution().sortByReflection([3, 5]))
assert wrong_32bit([3, 5]) == [5, 3] and Solution().sortByReflection([3, 5]) == [3, 5]

[5, 3] != [3, 5]


### Lỗi 3: so sánh chuỗi đảo theo từ điển thay vì theo giá trị số

`'1001'` nhỏ hơn `'111'` theo từ điển nhưng `9 > 7` theo số.

In [21]:
def wrong_lexicographic(nums):
    return sorted(nums, key=lambda x: (bin(x)[2:][::-1], x))          # SAI: khóa là chuỗi, không đổi sang số

print(wrong_lexicographic([7, 9]), "!=", Solution().sortByReflection([7, 9]))
assert wrong_lexicographic([7, 9]) == [9, 7] and Solution().sortByReflection([7, 9]) == [7, 9]

[9, 7] != [7, 9]


### Lỗi 4: đảo chữ số thập phân thay vì nhị phân

Đề nói **nhị phân**. Đảo chữ số thập phân `10 -> 01 -> 1` cho kết quả khác hẳn.

In [22]:
def wrong_decimal(nums):
    return sorted(nums, key=lambda x: (int(str(x)[::-1]), x))          # SAI: đảo thập phân

print(wrong_decimal([10, 6]), "!=", Solution().sortByReflection([10, 6]))
assert wrong_decimal([10, 6]) == [10, 6] and Solution().sortByReflection([10, 6]) == [6, 10]

[10, 6] != [6, 10]


### Lỗi 5: giữ `0b` khi đảo chuỗi

`bin(x)[::-1]` đảo cả `'0b'` nên `int(...)` báo lỗi. Luôn bỏ tiền tố bằng `[2:]` trước khi đảo.

In [23]:
try:
    int(bin(5)[::-1], 2)
except ValueError as e:
    print("lỗi đúng như dự đoán:", type(e).__name__)
assert int(bin(5)[2:][::-1], 2) == 5

lỗi đúng như dự đoán: ValueError


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp theo một đại lượng tự định nghĩa, hòa thì theo đại lượng khác" -> `sorted(key=lambda x: (đại_lượng, tiêu_chí_hòa))`.
- Nhị phân hóa: `bin(x)[2:]`; đảo: `[::-1]`; đọc lại: `int(s, 2)` (tự bỏ `0` đầu).
- Phản chiếu của `2x` bằng phản chiếu của `x`; nên nhiều giá trị hòa và tiêu chí hòa là bắt buộc.
- Nhớ `sorted` ổn định nhưng **không thay được** khóa hòa nếu đề yêu cầu thứ tự khác thứ tự ban đầu.
- Dùng khóa tuple thay `cmp_to_key` khi có thể: nhanh hơn và ít sai dấu.
- Tự kiểm tra: ví dụ `[6, 3]` phải ra `[3, 6]`.

## 15. Biến thể của bài

1. **Sort Integers by The Number of 1 Bits** (LeetCode 1356): sắp theo số bit `1`, hòa thì theo giá trị.
2. **Reverse Bits** (190): đảo bit của số 32 bit (giữ độ rộng) -- khác với phản chiếu ở bài này.
3. **Sort Array by Increasing Frequency** (1636): khóa `(tần suất, -giá trị)`.
4. **Relative Sort Array** (1122): sắp theo thứ tự tham chiếu.
5. **Sort the Jumbled Numbers** (2191): ánh xạ chữ số rồi sắp theo giá trị ánh xạ, hòa thì giữ thứ tự ban đầu.
6. Sắp xếp theo **tổng chữ số** hoặc **độ dài nhị phân** với cùng khung khóa tuple.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Insertion sort thủ công | so khóa, dịch và chèn | `O(n^2 log V)` | `O(n)` | Học thuật toán sắp xếp, `n` rất nhỏ |
| `sorted` + khóa chuỗi | `int(bin(x)[2:][::-1], 2)` | `O(n log n + n log V)` | `O(n)` | Nộp bài, ngắn gọn |
| `sorted` + khóa bit | vòng `while x` đảo bit | `O(n log n + n log V)` | `O(n)` | Luyện bit trick, tránh chuỗi |
| `cmp_to_key` | hàm so sánh tự viết | `O(n log n + n log V)` | `O(n)` | Tiêu chí phức tạp, không biểu diễn bằng khóa |

Nộp cách `sorted` + khóa chuỗi.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().sortByReflection([12, 3, 5, 9])` ra gì?

**Bài 2 (tính tay):** với `nums = [10, 1, 7, 2]`, tính phản chiếu từng số và sắp xếp.

**Bài 3 (điền TODO):** hoàn thành `reflect_str(x)` trả phản chiếu của `x` bằng chuỗi.

**Bài 4 (tự viết lại):** viết `sort_by_ones(nums)`: sắp theo số bit `1` tăng dần, hòa thì theo giá trị (bài LeetCode 1356).

**Bài 5 (mini challenge):** viết `group_by_reflection(nums)` trả về `dict` ánh xạ phản chiếu -> danh sách các số (đã sắp xếp tăng) có phản chiếu đó.

In [24]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().sortByReflection([12, 3, 5, 9]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def reflect_str_todo(x):
    # TODO: bin(x)[2:], đảo, rồi int(..., 2)
    return 0

[3, 12, 5, 9]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `12 = 1100 -> 0011 = 3`; `3 -> 3`; `5 -> 5`; `9 = 1001 -> 1001 = 9`. Bài 2: `10 = 1010 -> 0101 = 5`.

**Mức 2 (vừa):** Bài 3: ba bước `bin`, `[::-1]`, `int`. Bài 4: `bin(x).count('1')` làm khóa đầu. Bài 5: dùng `defaultdict(list)` rồi sắp từng danh sách.

**Mức 3 (gần đáp án):** Bài 1: phản chiếu `3, 3, 5, 9` -> hòa `3` giữa `3` và `12` nên `3` trước: `[3, 12, 5, 9]`. Bài 2: phản chiếu `10 -> 5`, `1 -> 1`, `7 -> 7`, `2 -> 1` -> `[1, 2, 10, 7]`. Bài 4: `[3, 5, 6]` đều có hai bit `1`, nên sắp theo giá trị.

## 19. Đáp án tham khảo cho bài tập

In [25]:
from collections import defaultdict

# Bài 1, 2
assert Solution().sortByReflection([12, 3, 5, 9]) == [3, 12, 5, 9]
assert Solution().sortByReflection([10, 1, 7, 2]) == [1, 2, 10, 7]

# Bài 3
def reflect_str(x):
    return int(bin(x)[2:][::-1], 2)
assert [reflect_str(v) for v in [4, 5, 3, 6, 8, 12, 9, 10]] == [1, 5, 3, 3, 1, 3, 9, 5]

# Bài 4
def sort_by_ones(nums):
    return sorted(nums, key=lambda x: (bin(x).count('1'), x))
assert sort_by_ones([0b0, 1, 2, 3, 4, 5, 6, 7, 8]) == [0, 1, 2, 4, 8, 3, 5, 6, 7]
assert sort_by_ones([1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1]) == [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

# Bài 5
def group_by_reflection(nums):
    d = defaultdict(list)
    for x in nums:
        d[reflect_str(x)].append(x)
    return {k: sorted(v) for k, v in d.items()}
g = group_by_reflection([3, 6, 5, 8, 12, 24])
assert g == {3: [3, 6, 12, 24], 5: [5], 1: [8]}
assert [x for k in sorted(g) for x in g[k]] == Solution().sortByReflection([3, 6, 5, 8, 12, 24])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp xếp theo đại lượng tự định nghĩa":**

```python
def f(x):                       # đại lượng cần sắp
    return int(bin(x)[2:][::-1], 2)

return sorted(nums, key=lambda x: (f(x), x))   # hòa thì theo giá trị gốc
```

**Checklist trước khi submit:**

- [ ] Phản chiếu tính trên chuỗi nhị phân **không** có `0b` và **không** có `0` đứng đầu?
- [ ] Khóa là tuple `(phản chiếu, giá trị)` (có tiêu chí hòa)?
- [ ] Không dùng độ rộng cố định 32 bit khi đảo?
- [ ] Không đảo chữ số thập phân nhầm?
- [ ] Thử `[6, 3]` (hòa), `[1]`, `[10**9]`, mảng có số trùng?
- [ ] Chữ ký đúng `sortByReflection(self, nums: List[int]) -> List[int]`?